<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 El Centro de Acopio de Papa: Qué es un Lago de Datos 🥔
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Big Data
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 02
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Big%20Data/02%20-%20Data%20Warehousing%20y%20Data%20Lake/Para%20Dummies/01_Data_Lake_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a aprender aquí? 🎈

Este cuaderno es la versión **«para no ingenieros»** del [cuaderno 01 del Módulo 02 (Data Lake, Hive y Pig)](../01_Data_Lake_Hive_y_Pig.ipynb). Piensa en un **centro de acopio de papa** en Boyacá: los campesinos llegan con **costales** de todos los tamaños y **no se les rechaza** nada en la puerta; después se **lava, se clasifica y se empaca** para vender. Un **lago de datos** funciona igual:

| En el centro de acopio | En el lago de datos |
|---|---|
| **Patio de recibo**: el costal se guarda **tal como llegó** (no se toca) | Zona **cruda** (bronce) |
| **Zona de lavado y clasificación**: se descartan las papas dañadas y se ordena por mes | Zona **curada** (plata) |
| **Bultos empacados** listos para el comprador | Zona **de negocio** (oro): resúmenes listos para usar |
| **Estantes rotulados por mes**: para buscar la papa de marzo no revisas todo el bodegón | **Particiones** (carpetas `mes=03`) |
| **El cuaderno del bodeguero**: qué hay en cada estante y dónde | El **catálogo** (*metastore*) |
| **Una receta paso a paso** («tome, escoja, pese, ordene, guarde») | **Pig Latin**; y **Hive** es «preguntarle en SQL a la bodega» |

Al terminar podrás explicar:
1. Las **tres zonas** y por qué la zona cruda **no se toca**.
2. Qué son las **particiones** y cómo **ahorran trabajo** (se abren menos archivos).
3. Para qué sirve el **catálogo** («el cuaderno del bodeguero»).
4. Cómo se ve una **receta estilo Pig** de punta a punta.

> 🔗 Si luego quieres ver el lago **con Parquet, Spark SQL, DuckDB** y un catálogo **hecho desde cero**: [cuaderno estándar](../01_Data_Lake_Hive_y_Pig.ipynb). Antes de esto: [el almacén de doña Rosa](00_Data_Warehousing_Dummies.ipynb).

---
## 1. Tres Zonas: Recibo, Lavado y Empaque 📦

Los campesinos entregan **un archivo cada tres días** con lo que llevaron (datos **inventados**). Como en la vida real, hay **costales con problemas**: unos kilos **negativos** (error al digitar) y un municipio **en blanco**. Hacemos tres cosas: (1) **recibir** todo en la zona cruda sin cambiarlo, (2) **lavar** pasando lo bueno a carpetas por **mes** y apartando lo malo **con su motivo**, y (3) **empacar** un resumen por municipio.

In [ ]:
import tempfile
from pathlib import Path
import numpy as np
import pandas as pd

carpeta = tempfile.TemporaryDirectory(prefix="acopio_")           # carpeta temporal: se borra al cerrar
B = Path(carpeta.name)
rng = np.random.default_rng(5)
municipios = ["Tunja", "Duitama", "Sogamoso", "Paipa"]

# --- 1) RECIBO: cada archivo se guarda TAL COMO LLEGO ---
(B / "cruda").mkdir()
for d in pd.date_range("2025-01-01", periods=90)[::3]:                     # un archivo cada 3 dias (30 archivos)
    n = int(rng.integers(3, 6))
    pd.DataFrame({"fecha": d.strftime("%Y-%m-%d"), "municipio": rng.choice(municipios, n), "kilos": rng.integers(20, 200, n)}).to_csv(
        B / "cruda" / f"costales_{d:%Y-%m-%d}.csv", index=False)
with open(B / "cruda" / "costales_2025-01-04.csv", "a") as f:
    f.write("2025-01-04,Tunja,-30\n")                                      # error al digitar: kilos negativos
with open(B / "cruda" / "costales_2025-02-03.csv", "a") as f:
    f.write("2025-02-03,,55\n")                                            # municipio en blanco
filas_crudas = sum(len(pd.read_csv(p, keep_default_na=False)) for p in (B / "cruda").glob("*.csv"))

# --- 2) LAVADO: lo bueno va a una carpeta por MES; lo malo se aparta con su motivo ---
apartados = []
for p in sorted((B / "cruda").glob("*.csv")):
    df = pd.read_csv(p, keep_default_na=False)
    malos = (df.kilos <= 0) | (df.municipio == "")
    for _, fila in df[malos].iterrows():
        apartados.append({"archivo": p.name, "motivo": "kilos negativos" if fila.kilos <= 0 else "municipio en blanco"})
    buenos = df[~malos]
    mes = p.stem[-5:-3]                                                    # '01', '02' o '03'
    destino = B / "curada" / f"mes={mes}"; destino.mkdir(parents=True, exist_ok=True)
    buenos.to_csv(destino / p.name, index=False)

curada = pd.concat(pd.read_csv(p) for p in sorted((B / "curada").rglob("*.csv")))
print(f"Recibo: entraron {filas_crudas} filas | pasaron a la zona curada {len(curada)} | se apartaron {len(apartados)}: {apartados}")
assert filas_crudas == len(curada) + len(apartados) and len(apartados) == 2                 # nada se pierde sin explicacion

# --- 3) EMPAQUE: un resumen listo para el comprador ---
oro = curada.groupby("municipio", as_index=False).kilos.sum().sort_values("kilos", ascending=False)
print("\nBulto empacado (kilos por municipio):"); print(oro.to_string(index=False))
assert oro.kilos.sum() == curada.kilos.sum()                                               # el resumen cuadra con la zona curada

---
**Lección:** la **zona cruda quedó intacta** (incluso con los dos errores): si mañana descubrimos que la regla de lavado estaba mal, **volvemos al patio de recibo y lavamos de nuevo**. La **cuenta cierra** (*lo que entró = lo que pasó + lo que se apartó*) y el **bulto empacado cuadra** con la zona curada. Esa disciplina es lo que evita que un lago se vuelva un **pantano**.

---

## 2. Estantes por Mes y el Cuaderno del Bodeguero 🗂️

Como la zona curada está en **carpetas por mes** (`mes=01`, `mes=02`, `mes=03`), si alguien pregunta *«¿cuántos kilos llegaron en marzo?»*, **solo hace falta abrir la carpeta de marzo**: es la **poda de particiones**. Y para saber **qué carpetas existen** sin recorrer toda la bodega, el bodeguero lleva un **cuaderno** (el **catálogo**): una lista de estantes con cuántos archivos tiene cada uno.

In [ ]:
def cuaderno_del_bodeguero(base):
    """El catalogo: para cada estante (carpeta mes=XX) anota cuantos archivos tiene."""
    return {d.name.split("=")[1]: sorted(d.glob("*.csv")) for d in sorted(base.glob("mes=*"))}


catalogo = cuaderno_del_bodeguero(B / "curada")
print("Cuaderno del bodeguero:", {mes: len(archivos) for mes, archivos in catalogo.items()})
todos = [a for archivos in catalogo.values() for a in archivos]
solo_marzo = catalogo["03"]
print(f"¿Kilos de marzo? Sin estantes habria que abrir {len(todos)} archivos; con el estante solo se abren {len(solo_marzo)}.")

kilos_con_poda = sum(pd.read_csv(a).kilos.sum() for a in solo_marzo)                          # solo el estante de marzo
kilos_sin_poda = sum(pd.read_csv(a).query("fecha >= '2025-03-01'").kilos.sum() for a in todos)   # revisando TODOS los archivos
print(f"Respuesta con estante: {kilos_con_poda} kilos | revisando todo: {kilos_sin_poda} kilos")
assert kilos_con_poda == kilos_sin_poda                                                       # misma respuesta, menos trabajo
assert len(solo_marzo) < len(todos) and len(todos) == 30 and sum(len(a) for a in catalogo.values()) == 30

---
**Lección:** con el estante rotulado **la respuesta es la misma** pero se abre **una fracción de los archivos** (los de marzo: ¡los otros dos meses ni se tocan!). El **cuaderno del bodeguero** es lo que permite decidir **qué abrir antes de abrir nada**. En un lago real hay miles de archivos y los datos están en la nube, así que **no abrir lo innecesario** ahorra tiempo y dinero. Una **precaución**: si los estantes se hacen **demasiado finos** (un archivo diminuto por cada día y por cada municipio), el bodeguero pierde más tiempo **abriendo cajitas** que sacando papa; es el «problema de los archivos pequeños».

---

## 3. La Receta Estilo Pig: Paso a Paso 🍲

**Pig** es una forma de dar instrucciones **como una receta**, con verbos fijos. **Hive**, en cambio, es **preguntar en SQL** («dame la suma por municipio»). Esta es una receta estilo Pig, con sus verbos, para *«los dos municipios con más kilos entre Tunja y Duitama»*:

| Verbo de Pig | Qué hace en la receta |
|---|---|
| `LOAD` | Traer los datos de los estantes |
| `FILTER` | Quedarse solo con Tunja y Duitama |
| `GROUP` + `FOREACH ... GENERATE` | Juntar por municipio y sumar los kilos |
| `ORDER` | Poner primero al que más trajo |
| `STORE` | Guardar el resultado |

In [ ]:
# LOAD: traer todo de los estantes
cargado = [fila for a in todos for fila in pd.read_csv(a).to_dict("records")]
# FILTER: solo Tunja y Duitama
filtrado = [f for f in cargado if f["municipio"] in ("Tunja", "Duitama")]
# GROUP: juntar los kilos de cada municipio
grupos = {}
for f in filtrado:
    grupos.setdefault(f["municipio"], []).append(f["kilos"])
# FOREACH ... GENERATE: sumar cada grupo
resumen = [{"municipio": m, "kilos": sum(ks), "costales": len(ks)} for m, ks in grupos.items()]
# ORDER: de mayor a menor
ordenado = sorted(resumen, key=lambda r: -r["kilos"])
# STORE: guardar el resultado
pd.DataFrame(ordenado).to_csv(B / "resultado_receta.csv", index=False)
print("Resultado de la receta:", ordenado)

# Comprobacion con pandas (otra forma de hacer lo mismo)
ref = curada[curada.municipio.isin(["Tunja", "Duitama"])].groupby("municipio").kilos.agg(["sum", "count"]).sort_values("sum", ascending=False)
assert [(r["municipio"], r["kilos"], r["costales"]) for r in ordenado] == [(m, int(v["sum"]), int(v["count"])) for m, v in ref.iterrows()]
assert (B / "resultado_receta.csv").exists()

---
**Lección:** la receta hace **lo mismo** que una línea de pandas, pero **escrita paso a paso**: eso era lo valioso de Pig cuando los datos eran tan grandes que había que repartirlos entre muchos computadores. **Honestidad:** hoy, para proyectos nuevos, Pig y Hive «a la antigua» han sido **reemplazados en gran medida por Spark** (que permite SQL o recetas con una sola herramienta y es más rápido); lo que **sigue vivo** son las **ideas** (zonas, estantes por mes, cuaderno del bodeguero). El estado exacto de cada proyecto conviene **verificarlo** en sus sitios oficiales.

---

## Resumen en una frase 🎯

> **Un lago de datos recibe todo tal cual llegó (zona cruda), lo lava y ordena en carpetas por mes (zona curada), prepara resúmenes listos para usar (zona de negocio) y lleva un cuaderno que dice qué hay y dónde, para abrir solo lo necesario.**

### Lo que aprendiste hoy:
- ✅ **Tres zonas:** cruda (intacta), curada (limpia, por mes) y de negocio (resúmenes); la cuenta **entró = pasó + se apartó** (124 = 122 + 2) debe cerrar.
- ✅ **Estantes por mes:** solo se abren los archivos del mes que se pregunta; **misma respuesta, menos trabajo**.
- ✅ **El cuaderno del bodeguero** (catálogo): sabe qué estantes existen **sin abrir** los archivos.
- ✅ **Receta estilo Pig** (`LOAD`, `FILTER`, `GROUP`, `FOREACH`, `ORDER`, `STORE`): paso a paso; hoy se hace sobre todo con **Spark**.

> 🎓 **Siguiente paso:** el [cuaderno estándar](../01_Data_Lake_Hive_y_Pig.ipynb) construye el lago con **Parquet**, un **catálogo con SQLite desde cero**, consultas con **Spark SQL y DuckDB** y mide la poda con datos de verdad. Luego: [bodega o plaza de mercado](02_DW_vs_Lake_Dummies.ipynb).

---
##### 🛠️ Práctica 1: ¿Cuántos kilos llegaron en febrero a Sogamoso?

Con el catálogo `catalogo` (un diccionario `mes -> lista de archivos`) hecho por `cuaderno_del_bodeguero`:

1. Abre **solo** los archivos del estante `"02"` y guarda en `febrero` un `DataFrame` con todos sus renglones (`pd.concat` de `pd.read_csv`).
2. Calcula en `kilos_sogamoso` cuántos kilos de **Sogamoso** hay en `febrero`.
3. Comprueba con `assert` que el resultado coincide con el que sale de la tabla completa `curada` filtrando `fecha` que empiece por `"2025-02"` y `municipio == "Sogamoso"`.

In [ ]:
# Escribe tu código aquí

# febrero = pd.concat(...)
# kilos_sogamoso = ...


<details>
<summary><b>💡 Haz clic aquí para ver la solución guiada...</b></summary>

```python
febrero = pd.concat(pd.read_csv(a) for a in catalogo["02"])
kilos_sogamoso = int(febrero[febrero.municipio == "Sogamoso"].kilos.sum())

esperado = int(curada[curada.fecha.str.startswith("2025-02") & (curada.municipio == "Sogamoso")].kilos.sum())
assert kilos_sogamoso == esperado
print("Kilos de Sogamoso en febrero:", kilos_sogamoso)
```
</details>

---


---
### 🧠 ¿Ya lo tienes claro?

1. ¿Por qué el centro de acopio **no rechaza** ningún costal en el patio de recibo? ¿Qué ventaja tiene guardar **también** lo malo (con su motivo)?
2. Explica con tus palabras cómo el «estante rotulado por mes» ahorra trabajo. ¿Qué pasaría si, en lugar de por mes, ordenaras los estantes por **el nombre del campesino** y la pregunta fuera por mes?
3. Si el cuaderno del bodeguero se **pierde** o queda **desactualizado**, ¿qué problemas aparecen? ¿Qué harías para mantenerlo al día?

---
<div align="center">
  <p style="font-size: 0.9em; color: #78350f;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Big Data (Edición Didáctica Para Dummies)</i>
  </p>
</div>